In [ ]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="",
    port=5432
)

cur = conn.cursor()

query = """
SELECT 
    user_session,
    product_id,
    price,
    event_type,
    event_time::TIMESTAMP as event_time,
    user_id
FROM makeup_consumer_events.dec
WHERE event_type IN ('cart', 'purchase')
ORDER BY user_session, event_time
"""

cur.execute(query)
results = cur.fetchall()

conn.close()

# 转成 DataFrame
df = pd.DataFrame(results, columns=['user_session', 'product_id', 'price', 'event_type', 'event_time', 'user_id'])
df = df.drop_duplicates(subset=['user_session', 'product_id', 'event_type'], keep='first')

print(f"提取了 {len(df)} 条数据")
#print(df.head(10))

提取了 1003376 条数据
                            user_session  product_id  price event_type  \
0   00002b0e-d7f7-454e-8386-431c4021a9f6     5891029   5.24       cart   
1   00002b0e-d7f7-454e-8386-431c4021a9f6     5890841   4.44       cart   
2   00002b0e-d7f7-454e-8386-431c4021a9f6     5622678   9.11       cart   
3   00002b0e-d7f7-454e-8386-431c4021a9f6     5622689   2.14       cart   
4   00002b0e-d7f7-454e-8386-431c4021a9f6     5645160   0.49       cart   
13  00002b0e-d7f7-454e-8386-431c4021a9f6     5650591   4.76       cart   
14  00002b0e-d7f7-454e-8386-431c4021a9f6     5774352   4.13       cart   
15  00002b0e-d7f7-454e-8386-431c4021a9f6     5686278   1.57       cart   
16  00002b0e-d7f7-454e-8386-431c4021a9f6     5818394   1.59       cart   
17  00002b0e-d7f7-454e-8386-431c4021a9f6     5877385   3.95       cart   

            event_time    user_id  
0  2019-12-18 06:50:18  531784651  
1  2019-12-18 06:50:37  531784651  
2  2019-12-18 06:52:53  531784651  
3  2019-12-18 06:54:17  5

In [ ]:
# 第二步：按 (user_session, product_id) 分组，合并信息
df_grouped = df.groupby(['user_session', 'product_id']).agg({
    'event_type': lambda x: list(x),
    'price': 'first',          # 同一对的价格相同，取第一个
    'user_id': 'first'         # 同一对的用户相同，取第一个
}).reset_index()  # ← 关键！把索引变回普通列

# 第三步：标记 A/B
df_grouped['group_type'] = df_grouped['event_type'].apply(
    lambda x: 'A' if 'purchase' in x else 'B'
)

# print(df_grouped.head(10))
print("\nA 组（购买）的行数:", len(df_grouped[df_grouped['group_type'] == 'A']))
print("B 组（未购买）的行数:", len(df_grouped[df_grouped['group_type'] == 'B']))

                           user_session  product_id event_type  price  \
0  00002b0e-d7f7-454e-8386-431c4021a9f6     5622678     [cart]   9.11   
1  00002b0e-d7f7-454e-8386-431c4021a9f6     5622689     [cart]   2.14   
2  00002b0e-d7f7-454e-8386-431c4021a9f6     5645160     [cart]   0.49   
3  00002b0e-d7f7-454e-8386-431c4021a9f6     5650591     [cart]   4.76   
4  00002b0e-d7f7-454e-8386-431c4021a9f6     5686278     [cart]   1.57   
5  00002b0e-d7f7-454e-8386-431c4021a9f6     5774352     [cart]   4.13   
6  00002b0e-d7f7-454e-8386-431c4021a9f6     5818394     [cart]   1.59   
7  00002b0e-d7f7-454e-8386-431c4021a9f6     5877385     [cart]   3.95   
8  00002b0e-d7f7-454e-8386-431c4021a9f6     5877489     [cart]   2.38   
9  00002b0e-d7f7-454e-8386-431c4021a9f6     5890841     [cart]   4.44   

     user_id group_type  
0  531784651          B  
1  531784651          B  
2  531784651          B  
3  531784651          B  
4  531784651          B  
5  531784651          B  
6  531784651  

In [5]:

results_df = df_grouped[df_grouped['price'] >= 0]
price_stats = results_df.groupby('group_type')['price'].agg([
    'count',    # 商品数
    'mean',     # 平均价格
    'median',   # 中位数
    'std',      # 标准差
    'min',
    'max'
]).round(2)  # 计算 A/B 组的平均统计指标

print(price_stats)

             count  mean  median    std  min     max
group_type                                          
A           211633  5.06    3.11   9.04  0.0  327.78
B           684580  5.38    3.65  10.21  0.0  327.78
